In [7]:
#!/usr/bin/env python3
"""
Analysis_insdn_notebook.py – Corrected version.
- Output directory is created before saving plots.
- CV scores are printed with 6 decimal places.
"""

import json
import os
import warnings
from typing import Dict, Tuple, Any, Optional

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    train_test_split,
)
from sklearn.preprocessing import StandardScaler

# Optional: %matplotlib inline

try:
    from xgboost import XGBClassifier
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

# ========== USER: Set these paths ==========
DATA_DIR = "."
OUTPUT_DIR = "./output"
# ===========================================

# ----------------------------------------------------------------------
# 1. Data loading
# ----------------------------------------------------------------------
def load_data(data_dir: str) -> pd.DataFrame:
    """Load and combine the three CSV files, preventing duplicate target columns."""
    files = {
        "Normal_data.csv": "normal",
        "OVS.csv": "attack",
        "metasploitable-2.csv": "attack",
    }

    dfs = []
    for filename, kind in files.items():
        filepath = os.path.join(data_dir, filename)
        if not os.path.isfile(filepath):
            raise FileNotFoundError(f"Required file not found: {filepath}")

        df = pd.read_csv(filepath)

        # Remove any existing columns that would conflict with our targets
        for col in ["label", "attack_type", "type"]:
            if col in df.columns and col not in ("type" if kind == "attack" else ""):
                df.drop(columns=[col], inplace=True)
        if "Label" in df.columns:
            df.rename(columns={"Label": "type"}, inplace=True)

        if kind == "normal":
            df["attack_type"] = "benign"
            df["label"] = 0
        else:
            if "type" not in df.columns:
                raise KeyError(f"No 'type' or 'Label' column found in {filename}.")
            df["attack_type"] = df["type"]
            df["label"] = 1

        dfs.append(df)

    combined = pd.concat(dfs, ignore_index=True)
    # Keep only numeric features plus our two target columns
    numeric_cols = combined.select_dtypes(include=[np.number]).columns.tolist()
    combined = combined.loc[:, ~combined.columns.duplicated()]
    target_cols = ["label", "attack_type"]
    keep_cols = [c for c in numeric_cols if c not in target_cols] + target_cols
    combined = combined[keep_cols]

    print(f"Loaded data: {combined.shape[0]} samples, {len(numeric_cols)} features.")
    print("Class distribution (label):")
    print(combined["label"].value_counts())
    print("\nAttack type distribution (attack_type):")
    print(combined["attack_type"].value_counts())

    return combined


# ----------------------------------------------------------------------
# 2. Preprocessing
# ----------------------------------------------------------------------
def preprocess(df: pd.DataFrame) -> Tuple[pd.DataFrame, np.ndarray, np.ndarray]:
    """Impute missing values and remove constant features."""
    feature_cols = [c for c in df.columns if c not in ("label", "attack_type")]
    X = df[feature_cols].copy()
    y = df["label"].values
    attack_types = df["attack_type"].values

    X = X.fillna(X.median())

    variances = X.var()
    constant_cols = variances[variances == 0].index.tolist()
    if constant_cols:
        print(f"Removing {len(constant_cols)} constant features.")
        X.drop(columns=constant_cols, inplace=True)

    return X, y, attack_types


# ----------------------------------------------------------------------
# 3. Binary models definition & training
# ----------------------------------------------------------------------
def get_binary_models() -> Dict[str, Tuple[Any, Dict[str, list]]]:
    """Return (estimator, param_grid) for each binary classifier."""
    models = {
        "LogisticRegression": (
            LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
            {"C": [0.1, 1.0, 10.0], "penalty": ["l2"], "solver": ["lbfgs", "liblinear"]},
        ),
        "RandomForest": (
            RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
            {"n_estimators": [100, 200], "max_depth": [None, 10, 20], "min_samples_split": [2, 5]},
        ),
    }

    if XGB_AVAILABLE:
        models["XGBoost"] = (
            XGBClassifier(random_state=RANDOM_STATE, use_label_encoder=False, eval_metric="logloss"),
            {"n_estimators": [100, 200], "learning_rate": [0.1, 0.01], "max_depth": [3, 5]},
        )
    else:
        models["GradientBoosting"] = (
            GradientBoostingClassifier(random_state=RANDOM_STATE),
            {"n_estimators": [100, 200], "learning_rate": [0.1, 0.01], "max_depth": [3, 5]},
        )

    return models


def train_binary_models(
    X_train: np.ndarray, y_train: np.ndarray, attack_types_train: np.ndarray
) -> Dict[str, Any]:
    """GridSearchCV with stratification on attack_type."""
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    cv_splits = list(skf.split(X_train, attack_types_train))
    models = get_binary_models()
    best_estimators = {}

    for name, (estimator, param_grid) in models.items():
        print(f"\n--- Training {name} (binary) ---")
        grid = GridSearchCV(
            estimator, param_grid, cv=cv_splits, scoring="f1", n_jobs=-1, verbose=1
        )
        grid.fit(X_train, y_train)
        best_estimators[name] = grid.best_estimator_
        print(f"Best parameters: {grid.best_params_}")
        print(f"Best CV F1: {grid.best_score_:.6f}")   # <-- More decimals

    return best_estimators


# ----------------------------------------------------------------------
# 4. Multiclass model training
# ----------------------------------------------------------------------
def train_multiclass_model(
    X_train_attack: np.ndarray, y_train_attack: np.ndarray
) -> RandomForestClassifier:
    """Train a RandomForest to predict attack type."""
    print("\n--- Training multiclass RandomForest ---")
    param_grid = {
        "n_estimators": [100, 200],
        "max_depth": [None, 10, 20],
        "min_samples_split": [2, 5],
    }
    rf = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    grid = GridSearchCV(rf, param_grid, cv=skf, scoring="f1_weighted", n_jobs=-1, verbose=1)
    grid.fit(X_train_attack, y_train_attack)
    print(f"Best parameters: {grid.best_params_}")
    print(f"Best CV weighted F1: {grid.best_score_:.6f}")   # <-- More decimals
    return grid.best_estimator_


# ----------------------------------------------------------------------
# 5. Evaluation & plots
# ----------------------------------------------------------------------
def evaluate_binary(model, X_test, y_test, model_name: str) -> Dict[str, float]:
    """Evaluate binary model, plot confusion matrix."""
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None

    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
    }
    if y_proba is not None:
        metrics["roc_auc"] = roc_auc_score(y_test, y_proba)

    # Print metrics with high precision (default dict print shows all decimals)
    print(f"\n{model_name} test metrics: {metrics}")

    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Normal", "Attack"], yticklabels=["Normal", "Attack"])
    plt.title(f"Confusion Matrix – {model_name}")
    plt.ylabel("True label")
    plt.xlabel("Predicted label")
    plt.tight_layout()
    save_path = os.path.join(OUTPUT_DIR, f"confusion_matrix_{model_name}.png")
    plt.savefig(save_path)
    plt.close()
    print(f"Confusion matrix saved as confusion_matrix_{model_name}.png")

    return metrics


def evaluate_multiclass(model, X_test, y_test, class_names) -> str:
    """Evaluate multiclass model, print report and plot confusion matrix."""
    y_pred = model.predict(X_test)
    report = classification_report(y_test, y_pred, target_names=class_names, zero_division=0)
    print("\nMulticlass classification report:\n", report)

    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.title("Confusion Matrix – Multiclass Attack Types")
    plt.ylabel("True label")
    plt.xlabel("Predicted label")
    plt.tight_layout()
    save_path = os.path.join(OUTPUT_DIR, "confusion_matrix_multiclass.png")
    plt.savefig(save_path)
    plt.close()
    print("Multiclass confusion matrix saved.")

    return report


# ----------------------------------------------------------------------
# 6. Save results
# ----------------------------------------------------------------------
def save_results(
    y_test, predictions, y_test_multiclass, pred_multiclass,
    binary_metrics, multiclass_report, models, output_dir
):
    """Write predictions.csv, metrics.json, and model files."""
    os.makedirs(output_dir, exist_ok=True)

    pred_df = pd.DataFrame({"true_label": y_test})
    for name, preds in predictions.items():
        pred_df[f"pred_{name}"] = preds
    if y_test_multiclass is not None:
        pred_df["true_attack_type"] = y_test_multiclass
        pred_df["pred_attack_type"] = pred_multiclass
    pred_df.to_csv(os.path.join(output_dir, "predictions.csv"), index=False)

    metrics_dict = {"binary": binary_metrics}
    if multiclass_report:
        metrics_dict["multiclass_report"] = multiclass_report
    with open(os.path.join(output_dir, "metrics.json"), "w") as f:
        json.dump(metrics_dict, f, indent=2)

    for name, model in models.items():
        joblib.dump(model, os.path.join(output_dir, f"{name}.joblib"))

    print(f"All results saved in {output_dir}")


# ----------------------------------------------------------------------
# Main pipeline
# ----------------------------------------------------------------------
def main():
    # Ensure output directory exists (fixes FileNotFoundError for early plots)
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # 1. Load and preprocess
    df = load_data(DATA_DIR)
    X, y, attack_types = preprocess(df)

    # 2. Stratified split
    X_train, X_temp, y_train, y_temp, at_train, at_temp = train_test_split(
        X, y, attack_types, test_size=0.4, stratify=attack_types, random_state=RANDOM_STATE
    )
    X_val, X_test, y_val, y_test, at_val, at_test = train_test_split(
        X_temp, y_temp, at_temp, test_size=0.5, stratify=at_temp, random_state=RANDOM_STATE
    )
    print(f"\nTrain: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

    # 3. Scaling
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    # 4. Binary models
    binary_models = train_binary_models(X_train_scaled, y_train, at_train)

    binary_metrics = {}
    predictions_binary = {}
    for name, model in binary_models.items():
        metrics = evaluate_binary(model, X_test_scaled, y_test, name)
        binary_metrics[name] = metrics
        predictions_binary[name] = model.predict(X_test_scaled)

    # 5. Multiclass (attack samples only)
    train_attack_mask = y_train == 1
    X_train_attack = X_train_scaled[train_attack_mask]
    y_train_attack = at_train[train_attack_mask]

    test_attack_mask = y_test == 1
    X_test_attack = X_test_scaled[test_attack_mask]
    y_test_attack = at_test[test_attack_mask]

    multiclass_model = train_multiclass_model(X_train_attack, y_train_attack)

    if len(np.unique(y_test_attack)) > 0:
        class_names = sorted(np.unique(y_test_attack))
        multiclass_report = evaluate_multiclass(multiclass_model, X_test_attack, y_test_attack, class_names)
        pred_multiclass = multiclass_model.predict(X_test_attack)
    else:
        multiclass_report = None
        pred_multiclass = None
        print("No attack samples in test set – skipping multiclass evaluation.")

    # 6. Save all
    if pred_multiclass is not None:
        full_pred_multiclass = np.full(len(y_test), np.nan, dtype=object)
        full_pred_multiclass[test_attack_mask] = pred_multiclass
    else:
        full_pred_multiclass = None

    save_results(
        y_test=y_test,
        predictions=predictions_binary,
        y_test_multiclass=at_test,
        pred_multiclass=full_pred_multiclass,
        binary_metrics=binary_metrics,
        multiclass_report=multiclass_report,
        models={**binary_models, "multiclass_RF": multiclass_model},
        output_dir=OUTPUT_DIR,
    )

    print("\nPipeline completed successfully.")


if __name__ == "__main__":
    main()

Loaded data: 343889 samples, 80 features.
Class distribution (label):
label
1    275465
0     68424
Name: count, dtype: int64

Attack type distribution (attack_type):
attack_type
Probe         98129
DDoS          73529
benign        68424
DoS           53616
DDoS          48413
BFA            1405
Web-Attack      192
BOTNET          164
U2R              17
Name: count, dtype: int64
Removing 11 constant features.

Train: 206333, Val: 68778, Test: 68778

--- Training LogisticRegression (binary) ---
Fitting 3 folds for each of 6 candidates, totalling 18 fits
Best parameters: {'C': 10.0, 'penalty': 'l2', 'solver': 'liblinear'}
Best CV F1: 0.998103

--- Training RandomForest (binary) ---
Fitting 3 folds for each of 12 candidates, totalling 36 fits
Best parameters: {'max_depth': 20, 'min_samples_split': 5, 'n_estimators': 100}
Best CV F1: 0.999903

--- Training XGBoost (binary) ---
Fitting 3 folds for each of 8 candidates, totalling 24 fits
Best parameters: {'learning_rate': 0.1, 'max_depth'